<small>📌 <b>먼저 내 노트로 저장하기</b> — ① 맨 위 <b>파일 → Drive에 사본 저장</b> (안 하면 쓴 게 사라져요) ② 사본 오른쪽 위 <b>공유</b> → 일반 액세스 「링크가 있는 모든 사용자」 → <b>링크 복사</b> → 수행평가 화면 「📓 내 Colab 실습 노트」 칸에 붙여넣고 보관.</small><br><img src="https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/colab_save.png" width="300"> <img src="https://raw.githubusercontent.com/SoojiNoh/baehwa-colab/main/nb/img/colab_share.png" width="300">

# 🧠 내가 정한 주제 — 딥러닝(신경망) 빈칸 실습 (내 CSV 로)

결과 열이 **0 또는 1**(예: 생존/아님, 합격/불합격)인 데이터로 신경망 분류를 해요. 결과가 숫자 크기이면 아래 2-1 에서 「평균보다 큰가」로 0/1 을 만들어요.

수업 「#20 과일 채소 딥러닝」·「MNIST 딥러닝」 순서 그대로: 올리기 → 거르기 → 정규화 → 신경망 설계(Sequential) → compile → fit → evaluate.

`______` 빈칸을 채우고 ▶ 실행해요. 막히면 **오픈북** — 클래스룸 실습 파일.

In [ ]:
# 0. 준비 — 그대로 실행해요 (한글 그래프 글꼴)
!pip -q install koreanize-matplotlib
import matplotlib.pyplot as plt
import koreanize_matplotlib

## 1. 내 데이터 올리기
내가 찾은 CSV 파일을 올려요. ▶ 실행하면 「파일 선택」 단추가 나와요.

- 엑셀 파일이면 먼저 「다른 이름으로 저장 → CSV UTF-8」로 바꿔요.
- 구글 스프레드시트면 「파일 → 다운로드 → CSV」로 받아 올려요.
- 공공데이터포털(data.go.kr)·KOSIS 에서 받은 CSV 가 한글이 깨지면 아래 `encoding="cp949"` 줄을 써요.

In [ ]:
from google.colab import files
import pandas as pd
up = files.upload()                    # 파일 선택 창이 떠요
name = next(iter(up))
try:
    df = pd.______(name)               # 힌트: CSV 를 읽는 함수
except UnicodeDecodeError:
    df = pd.read_csv(name, encoding="cp949")      # 한글 깨짐(공공데이터) 대비
print(df.shape)
df.______()                            # 힌트: 앞 5줄만 보기

## 2. 열 살펴보고 내 주제에 맞게 고르기
아래 목록에서 **결과(예측하고 싶은 것)** 열 하나와 **원인일 것 같은 숫자 열** 몇 개를 골라 이름을 그대로 적어요.

In [ ]:
df.info()      # 열 이름·빈칸 아닌 개수·자료형(숫자는 int/float)

In [ ]:
Y = "결과 열 이름"                    # ← 바꿔요: 예측하고 싶은 열
X_COLS = ["원인 열1", "원인 열2"]      # ← 바꿔요: 숫자 열 몇 개
for c in [Y] + X_COLS:
    assert c in df.columns, f"'{c}' 라는 열이 없어요 — 위 목록의 이름을 그대로 복사해요"
df = df[[Y] + X_COLS]
for c in df.columns:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', ''), errors='coerce')      # '1,234' 같은 글자 숫자를 숫자로
df.head()

## 3. 결측치(빈칸) 처리
📖 **오픈북** — 수업 결측치 실습 `isnull().sum()`, `dropna()`

In [ ]:
print(df.______().sum())      # 힌트: 빈칸인지 True/False — 열마다 개수
df = df.______()              # 힌트: 빈칸 있는 줄 지우기
print('빈칸 지운 뒤', df.shape)

## 4. 이상치 거르기 (IQR)
Q1(25%)·Q3(75%) 사이 폭(IQR)의 1.5배를 넘게 벗어난 값은 이상치로 보고 걸러요. 값 종류가 적은 열(0/1 등)은 건너뛰어요.

In [ ]:
for col in df.columns:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(______)      # 힌트: 75% 는 소수로?
    iqr = q3 - q1
    if iqr == 0 or df[col].nunique() <= 5: continue
    df = df[(df[col] >= q1 - 1.5 * iqr) & (df[col] <= q3 + 1.5 * iqr)]
print('거른 뒤', df.shape)

## 2-1. 결과를 0/1 로 (이미 0/1 이면 그대로)

In [ ]:
if df[Y].nunique() > 2:
    df[Y] = (df[Y] > df[Y].mean()).astype(int)      # 평균보다 크면 1
print(df[Y].value_counts())

## 2-2. 속성 살펴보기 — pairplot · 상관 heatmap
신경망에 넣기 전에 열끼리 어떤 관계인지 눈으로 봐요(그대로 실행). 색은 결과(0·1)예요 — 두 색 점이 잘 갈라지는 열일수록 신경망이 배우기 쉬워요. 📖 **오픈북** — 「#16 펭귄-분류」 `sns.pairplot` · 「#14 Iris 데이터 전처리」 `sns.heatmap`

In [ ]:
import seaborn as sns
sns.pairplot(df[X_COLS + [Y]].sample(min(len(df), 300), random_state=0), hue=Y, corner=True, plot_kws={'s': 12})
plt.show()
plt.figure(figsize=(7, 5))
sns.heatmap(df[X_COLS + [Y]].corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

## 5. 나누고 0~1 로 맞추기(정규화)
📖 **오픈북** — 「MNIST 딥러닝」 정규화

In [ ]:
from sklearn.model_selection import train_test_split
X, y = df[X_COLS], df[Y]
X_train, X_test, y_train, y_test = ______(X, y, test_size=0.3, random_state=42)      # 힌트: 나누는 함수
mn, mx = X_train.min(), X_train.max()
rng = (mx - mn).replace(0, 1)
X_train, X_test = (X_train - mn) / rng, (X_test - mn) / rng

## 6. 신경망 설계 — 수업 문법 그대로
📖 **오픈북** — 「#20 과일 채소 딥러닝」 `keras.Sequential([...])`

In [ ]:
from tensorflow import keras
model = keras.______([                                  # 힌트: 층을 차례로 쌓는 모델
    keras.layers.Input(shape=(len(X_COLS),)),
    keras.layers.Dense(16, activation="______"),          # 힌트: 은닉층에 쓰는 활성화 함수(r…)
    keras.layers.Dense(1, activation="sigmoid")])        # 0~1 확률
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## 7. 학습·평가

In [ ]:
hist = model.______(X_train, y_train, epochs=50, validation_split=0.2, verbose=0)      # 힌트: 학습시키기
loss, acc = model.______(X_test, y_test, verbose=0)      # 힌트: 시험 보기
print('시험 정확도:', round(acc, 3))
plt.plot(hist.history['accuracy'], label='학습'); plt.plot(hist.history['val_accuracy'], label='검증'); plt.legend(); plt.show()

## 8. 내 생각
- 정확도 ___ — 처음 학습용 1 의 비율(위 value_counts)보다 ___ (높다/비슷하다).
- 더 좋게 하려면 ___ 열을 더 넣거나 epochs 를 ___ 해 보겠다.